# Домашнее задание №15. Keras

## Задание 1
1.Самостоятельно выбранными средствами (opencv, pillow (PIL), …) сгенерировать по 820 картинок размером 100х100 пикселей (px) для каждой из цифр: 0, 1, 3, 8 следующим образом (800 – тренировочная выборка, 20 – тестовая выборка № 1):
 фон картинки белый,\
    1. цифра: ширина – 20 px, высота – 50 px, цвет линии – черный, цифра целиком помещается в картинку, цифра находится в случайном месте на картинке,\
    2. на изображении цифра расположена так, что ее вертикальная ось параллельна оси ординат (вертикальное положение) или оси абсцисс (горизонтальное положение),\
    3. тренировочная выборка содержит 400 изображений каждой цифры в горизонтальном положении и 400 изображений каждой цифры в вертикальном положении,\
    4. тестовая выборка содержит 10 изображений каждой цифры в горизонтальном положении и 10 изображений каждой цифры в вертикальном положении.

2.Создать новые тестовые картинки, полученные путем добавления черных пикселей (шум) в случайно выбранные места сгенерированных тестовых картинок:\
    1. 20 пикселей (тестовая выборка № 2),\
    2. 50 пикселей (тестовая выборка № 3),\
    3. 100 пикселей (тестовая выборка № 4),\
    4. 200 пикселей (тестовая выборка № 5).

In [ ]:
import os
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import random

# Настройки
BASE_DIR = "data"
IMAGE_SIZE = 100
DIGIT_WIDTH, DIGIT_HEIGHT = 20, 50
WHITE = (255, 255, 255)
BLACK = (0, 0, 0)
NOISE_LEVELS = {'test2': 20, 'test3': 50, 'test4': 100, 'test5': 200}

def create_dirs():
    digits = ['0', '1', '3', '8']
    sets = ['train', 'test1'] + list(NOISE_LEVELS.keys())
    for digit in digits:
        for set_name in sets:
            os.makedirs(os.path.join(BASE_DIR, set_name, digit), exist_ok=True)

def get_font():
    try:
        return ImageFont.truetype("arial.ttf", 40)
    except IOError:
        return ImageFont.load_default()

def generate_digit_image(digit, orientation):
    # 1. Создаём финальный белый фон 100x100
    bg = Image.new('RGB', (IMAGE_SIZE, IMAGE_SIZE), WHITE)
    
    # 2. Создаём промежуточный слой с ПРОЗРАЧНЫМ фоном для отрисовки цифры
    # Используем режим 'RGBA', где 4-й канал - прозрачность
    tmp_layer = Image.new('RGBA', (60, 80), (0, 0, 0, 0))
    draw = ImageDraw.Draw(tmp_layer)
    
    # Загружаем шрифт
    font = get_font()
    
    # Рисуем цифру ЧЕРНЫМ цветом с полной непрозрачностью (255)
    draw.text((0, 0), digit, font=font, fill=(0, 0, 0, 255))
    
    # 3. Обрезаем лишнее пространство вокруг цифры (по контенту)
    bbox = tmp_layer.getbbox()
    if bbox is None:
        return bg
    digit_crop = tmp_layer.crop(bbox)
    
    # 4. Масштабируем цифру до СТРОГО 20x50 пикселей
    digit_resized = digit_crop.resize((DIGIT_WIDTH, DIGIT_HEIGHT), Image.Resampling.LANCZOS)
    
    # 5. Поворачиваем ТОЛЬКО слой с цифрой, если нужно
    if orientation == 'vertical':
        # fillcolor=(0,0,0,0) важен: заполняем новые углы прозрачностью, а не белым/черным
        digit_final = digit_resized.rotate(90, expand=True, fillcolor=(0, 0, 0, 0))
    else:
        digit_final = digit_resized
        
    # 6. Вычисляем случайную позицию для вставки на фон 100x100
    max_x = IMAGE_SIZE - digit_final.width
    max_y = IMAGE_SIZE - digit_final.height
    x = random.randint(0, max_x)
    y = random.randint(0, max_y)
    
    # 7. Вставляем цифру на фон, используя альфа-канал как маску
    # Это гарантирует, что прозрачные пиксели не закрасят белый фон
    bg.paste(digit_final, (x, y), digit_final)
    
    return bg

def add_noise(img, noise_pixels):
    arr = np.array(img)
    h, w = arr.shape[:2]
    # Гарантируем точное количество УНИКАЛЬНЫХ пикселей
    total_pixels = h * w
    count = min(noise_pixels, total_pixels)
    indices = np.random.choice(total_pixels, count, replace=False)
    arr.flat[indices] = BLACK
    return Image.fromarray(arr)

def generate_dataset():
    digits = ['0', '1', '3', '8']
    orientations = ['horizontal', 'vertical']
    
    create_dirs()
    
    # 1. Генерация train и test1
    for digit in digits:
        for orientation in orientations:
            # Тренировочная выборка: 400 на ориентацию
            for i in range(400):
                img = generate_digit_image(digit, orientation)
                img.save(os.path.join(BASE_DIR, 'train', digit, f'{digit}_{orientation}_{i:03d}.png'))
                
            # Тестовая выборка №1: 10 на ориентацию
            for i in range(10):
                img = generate_digit_image(digit, orientation)
                img.save(os.path.join(BASE_DIR, 'test1', digit, f'{digit}_{orientation}_{i:03d}.png'))

    # 2. Генерация зашумленных тестовых выборок
    for digit in digits:
        for orientation in orientations:
            for i in range(10):
                src_path = os.path.join(BASE_DIR, 'test1', digit, f'{digit}_{orientation}_{i:03d}.png')
                original_img = Image.open(src_path)
                
                for test_name, pixels in NOISE_LEVELS.items():
                    noisy_img = add_noise(original_img.copy(), pixels)
                    noisy_img.save(os.path.join(BASE_DIR, test_name, digit, f'{digit}_{orientation}_{i:03d}.png'))

if __name__ == "__main__":
    generate_dataset()
    print("Все изображения успешно сгенерированы!")

Все изображения успешно сгенерированы!


# Задание №2

Не используя предобученные модели (сети), модифицировать скрипт задачи «Dogs vs Cats» (с семинара) или написать свою нейронную сеть на keras такую, что:

1) На вход подается тренировочное множество: по 800 картинок каждой цифры.
2) Из тренировочного множества выделяется часть картинок (10-20%), на валидационное множество, в котором должны присутствовать цифры в вертикальном и горизонтальном положении.
3) Протестировать адекватность модели на всех тестовых выборках № 1, № 2, № 3, № 4, № 5, фиксируя при этом точность (accuracy) классификации.
4) Повторить пункты 1)–3), изменив объем тренировочной выборки до 600, 400, 200, 100 картинок каждой цифры.

<i> Могут пригодиться <tt>Dense</tt>, <tt>Conv2D</tt>, <tt>MaxPooling2D</tt>, <tt>Flatten</tt>.</i>

Результаты оформить в виде таблицы со столбцами: размер тренировочной выборки, количество шумовых пикселей, точность (accuracy) классификации.

In [ ]:
import os
import numpy as np
import pandas as pd
from PIL import Image
import shutil
import random

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.utils import set_random_seed


# MaxPooling2D -- Берёт максимальное значение из каждого окна
# AveragePooling2D -- Вычисляет среднее арифметическое всех значений в окне
# GlobalAveragePooling2D / GlobalMaxPooling2D -- Окно пулинга равно размеру всей карты признаков (H, W). На выходе получается вектор длиной C (число каналов)
# Overlapping Pooling -- strides < pool_size. Например, окно 3×3, шаг 2.
# Stochastic Pooling -- Элемент из окна выбирается случайно, но вероятность выбора пропорциональна его значению
# === Настройки ===
BASE_DIR = "data"
IMG_SIZE = 100
BATCH_SIZE = 32
EPOCHS = 30
CLASSES = ['0', '1', '3', '8']
ORIENTATIONS = ['horizontal', 'vertical']
NOISE_LEVELS = {'test1': 0, 'test2': 20, 'test3': 50, 'test4': 100, 'test5': 200}
RANDOM_SEED = 42

set_random_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)


def create_model():
    """Простая CNN без предобученных весов"""
    model = Sequential([
        # 32 - количество фильтров (свёрточных ядер). Каждый фильтр ищет определённый паттерн (края, текстуры, формы).
        # (3, 3) - размер ядра свёртки (высота на ширина). Окно размером 3 на 3 пикселя скользит по изображению.
        # activation='relu' - функция активации ReLU. Обнуляет отрицательные значения, ускоряет обучение и помогает модели выучивать нелинейные зависимости.
        # input_shape=(IMG_SIZE, IMG_SIZE, 3) - форма входного тензора. IMG_SIZE - высота и ширина изображения, 3 - цветовые каналы (RGB).
        Conv2D(32, (3, 3), activation='relu', input_shape=(IMG_SIZE, IMG_SIZE, 3)),
        # Слой берёт максимальное значение из каждого блока 2 на 2, уменьшая пространственные размеры карты признаков ровно в 2 раза
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        Conv2D(64, (3, 3), activation='relu'),
        MaxPooling2D(2, 2),
        # Преобразует трёхмерный тензор (высота на ширина на каналы) в одномерный вектор. 
        Flatten(),
        # количество нейронов в полносвязном (скрытом) слое
        Dense(128, activation='relu'),
        # доля случайно "отключаемых" нейронов во время обучения (50%)
        Dropout(0.5),
        # количество выходных нейронов, равное числу целевых классов
        # преобразует "сырые" выходы (логиты) в вероятности
        Dense(len(CLASSES), activation='softmax')
    ])

    # categorical_crossentropy -- one-hot, многоклассовая классификация, сравнивает распределения вероятностей.
    # sparse_categorical_crossentropy -- целые числа(массив)
    # binary_crossentropy -- бинарные(0,1) или один столбец вероятностей
    # mse / mae -- вещественные числа, используется регрессия (предсказание цены, координат, температуры)
    # kl_divergence -- распределения вероятностей, Штрафует расхождение между предсказанным и целевым распределением
    model.compile(optimizer='adam',
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Batch Normalization - нормализация по батчу
# Weight Decay - Это эквивалентно L2(Loss_данных + λ · Σ(w²)), но применяется на этапе обновления весов
# Data Augmentation (регуляризация на уровне данных)
#  - При каждой эпохе изображения проходят через случайные преобразования: поворот, отражение, масштабирование, изменение яркости/контраста, сдвиг.
#  - Модель никогда не видит одно и то же изображение дважды в одинаковом виде.
# Early Stopping - Останавливает обучение до переобучения(регуляризация по времени)


def prepare_stratified_subset(source_dir, target_dir, images_per_digit, orientations):
    """
    Копирует ровно images_per_digit картинок на цифру,
    гарантируя баланс по ориентациям (половина horizontal, половина vertical).
    """
    os.makedirs(target_dir, exist_ok=True)
    
    for digit in CLASSES:
        digit_dir = os.path.join(source_dir, digit)
        target_digit_dir = os.path.join(target_dir, digit)
        os.makedirs(target_digit_dir, exist_ok=True)
        
        # Разделяем файлы по ориентациям
        files_by_orient = {orient: [] for orient in orientations}
        for fname in os.listdir(digit_dir):
            if not fname.endswith('.png'):
                continue
            for orient in orientations:
                if orient in fname:
                    files_by_orient[orient].append(fname)
                    break
        
        # Берем поровну из каждой ориентации
        per_orient = images_per_digit // len(orientations)
        selected = []
        for orient in orientations:
            random.shuffle(files_by_orient[orient])
            selected.extend(files_by_orient[orient][:per_orient])
        
        # Копируем
        for fname in selected:
            shutil.copy2(
                os.path.join(digit_dir, fname),
                os.path.join(target_digit_dir, fname)
            )


def load_data(train_dir, test_dirs, validation_split=0.2):
    """Загружает данные с аугментацией для тренировки и без для тестов"""
    # Для тренировки: аугментация + валидация
    train_datagen = ImageDataGenerator(
        rescale=1./255,
        validation_split=validation_split,
        rotation_range=15,
        width_shift_range=0.1,
        height_shift_range=0.1,
        zoom_range=0.1
    )
    
    train_gen = train_datagen.flow_from_directory(
        train_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='training',
        seed=RANDOM_SEED,
        shuffle=True
    )
    
    val_gen = train_datagen.flow_from_directory(
        train_dir,
        target_size=(IMG_SIZE, IMG_SIZE),
        batch_size=BATCH_SIZE,
        class_mode='categorical',
        subset='validation',
        seed=RANDOM_SEED,
        shuffle=False
    )

    test_gens = {}
    test_datagen = ImageDataGenerator(rescale=1./255)
    for test_name, test_path in test_dirs.items():
        if not os.path.exists(test_path):
            continue
        test_gens[test_name] = test_datagen.flow_from_directory(
            test_path,
            target_size=(IMG_SIZE, IMG_SIZE),
            batch_size=BATCH_SIZE,
            class_mode='categorical',
            shuffle=False
        )
    
    return train_gen, val_gen, test_gens


def train_and_evaluate(train_sizes_per_digit):
    """Основной цикл эксперимента"""
    results = []
    base_train = os.path.join(BASE_DIR, 'train')
    test_dirs = {name: os.path.join(BASE_DIR, name) for name in NOISE_LEVELS.keys()}
    
    for size in train_sizes_per_digit:
        print(f"\n=== Обучение на {size} изображений на цифру ===")
        
        # 1. Подготовка стратифицированной подвыборки
        subset_dir = os.path.join(BASE_DIR, f'train_{size}')
        prepare_stratified_subset(base_train, subset_dir, size, ORIENTATIONS)
        
        # 2. Загрузка данных
        train_gen, val_gen, test_gens = load_data(subset_dir, test_dirs)
        
        # 3. Создание и обучение модели
        model = create_model()
        early_stop = EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1)
        
        history = model.fit(
            train_gen,
            validation_data=val_gen,
            epochs=EPOCHS,
            callbacks=[early_stop],
            verbose=1
        )
        
        # 4. Тестирование на всех наборах
        for test_name, test_gen in test_gens.items():
            if test_gen is None:
                continue
            test_gen.reset()  # Сброс генератора для корректной оценки
            # финальная оценка обобщающей способности модели на тестовых данных
            _, acc = model.evaluate(test_gen, verbose=0)
            results.append({
                'Размер тренировочной выборки (на цифру)': size,
                'Количество шумовых пикселей': NOISE_LEVELS[test_name],
                'Точность (accuracy)': f"{acc:.4f}"
            })
            print(f"  {test_name} (шум {NOISE_LEVELS[test_name]}px): accuracy = {acc:.4f}")
    
    return pd.DataFrame(results)
    

def main():
    # Размеры выборки: количество изображений НА ОДНУ цифру
    train_sizes = [800, 600, 400, 200, 100]
    
    results_df = train_and_evaluate(train_sizes)
    
    # Вывод и сохранение
    print("\n" + "="*60)
    print("РЕЗУЛЬТАТЫ:")
    print(results_df.to_string(index=False))
    results_df.to_csv('classification_results.csv', index=False, encoding='utf-8-sig')
    print(f"\nРезультаты сохранены в 'classification_results.csv'")


if __name__ == "__main__":
    main()


=== Обучение на 800 изображений на цифру ===
Found 2560 images belonging to 4 classes.
Found 640 images belonging to 4 classes.
Found 80 images belonging to 4 classes.
Found 80 images belonging to 4 classes.
Found 80 images belonging to 4 classes.
Found 80 images belonging to 4 classes.
Found 80 images belonging to 4 classes.


C:\Users\reino\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\layers\convolutional\base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
C:\Users\reino\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\trainers\data_adapters\py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 15s 170ms/step - accuracy: 0.2949 - loss: 1.3710 - val_accuracy: 0.5641 - val_loss: 0.9334
Epoch 2/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 5s 68ms/step - accuracy: 0.5958 - loss: 0.8655 - val_accuracy: 0.8453 - val_loss: 0.4847
Epoch 3/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 5s 68ms/step - accuracy: 0.8048 - loss: 0.4588 - val_accuracy: 0.9422 - val_loss: 0.2255
Epoch 4/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 5s 67ms/step - accuracy: 0.9078 - loss: 0.2441 - val_accuracy: 0.9641 - val_loss: 0.1178
Epoch 5/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 6s 68ms/step - accuracy: 0.9573 - loss: 0.1400 - val_accuracy: 0.9797 - val_loss: 0.0729
Epoch 6/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 5s 68ms/step - accuracy: 0.9417 - loss: 0.1497 - val_accuracy: 0.9828 - val_loss: 0.0516
Epoch 7/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 5s 68ms/step - accuracy: 0.9766 - loss: 0.0672 - val_accuracy: 0.9875 - val_loss: 0.0419
Epoch 8/30
80/80 ━━━━━━━━━━━━━━━━━━━━ 6s 68ms/step - accuracy: 0.9788 - loss: 0.0587 - val_accuracy: 0.9922 -